In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

# 使用python-dotenv
from dotenv import load_dotenv
load_dotenv("/workspace/data/.env")
import os
print(os.getenv("DEEPSEEK_API_KEY"))

sk-b0010ce810bc4604b1802762832f8acd


In [2]:
from service.db_service import DBService

# Get the singleton instance (connects on first call)
db_service = DBService()

# Get the database object
db = db_service.get_db()
print(db.list_collection_names())

# Get a specific collection
plugin_outputs_collection = db_service.get_collection("plugin_outputs")

# Use the collection...
build_count = plugin_outputs_collection.count_documents({})
print(f"Found {build_count} outputs.")


# Close connection when application exits (optional, depends on app lifecycle)
# db_service.close_connection()

2025-04-10 15:35:52,653 - service.db_service - INFO - Successfully connected to MongoDB: mongodb/cfed?authSource=cfed


['builds', 'plugin_outputs']
Found 226 outputs.


In [3]:
import json

# Retrieve a single record from the collection
record = plugin_outputs_collection.aggregate([{ "$sample": { "size": 1 } }]).next()


db_service.close_connection()



2025-04-10 15:35:54,620 - service.db_service - INFO - MongoDB connection closed.


In [9]:
from model.data_model import DataModel
# from model.augmented_data_model import AugmentedDataModel
# from model.distillation_models import KnowledgeDistillation
from data_util.data_augment_service import DataAugmentService
from model.augmented_data_model import AugmentType
from data_util.key_info_extraction import extract_core_instructions
data = DataModel.from_dict(record)
# print(data)

# Find the RTL_protected file content
rtl_file = None
for filename, file_data in data.plugin_files.items():
    if "RTL_Protected" in filename:
        rtl_file = file_data
        break
code = rtl_file.content if rtl_file else None
# Convert bytes to string if needed
if isinstance(code, bytes):
    code = code.decode('utf-8')
print(code)

cleaned_code = extract_core_instructions(code)


augment_service = DataAugmentService()

augmented_content = augment_service.augment_code(cleaned_code, AugmentType.DUPLICATE_BLOCKS)
print(augmented_content)

# augmented_data = AugmentedDataModel.from_data_model(data)



BB: -2

----------------------------------------------------------------

BB: 0
(insn 110 3 109 2 (parallel [
            (asm_input/v ("STMDB r6!, {r11}") <built-in>:0)
            (clobber (mem:BLK (scratch) [0  A8]))
        ]) -1
     (nil))
(insn 109 110 92 2 (set (reg:SI 11 fp)
        (const_int 560 [0x230])) -1
     (nil))
(insn 92 109 93 2 (set (reg:SI 11 fp)
        (plus:SI (reg:SI 11 fp)
            (const_int -353 [0xfffffffffffffe9f]))) -1
     (nil))
(insn 93 92 94 2 (set (reg:CC 100 cc)
        (compare:CC (reg:SI 11 fp)
            (const_int 207 [0xcf]))) -1
     (nil))
(insn 94 93 66 2 (set (pc)
        (if_then_else (ne (reg:CC 100 cc)
                (const_int 0 [0]))
            (label_ref 90)
            (pc))) -1
     (nil))
(insn/f 66 94 67 (parallel [
            (set (mem/c:BLK (pre_modify:SI (reg/f:SI 13 sp)
                        (plus:SI (reg/f:SI 13 sp)
                            (const_int -8 [0xfffffffffffffff8]))) [24  A8])
                (unspec:B

/workspace/src/analyzer/service/llm_service.py:20: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = deepseek_chat(messages)
2025-04-10 15:48:45,675 - httpx - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


BB: -2
BB: 0
(insn (parallel [
            (asm_input/v ("STMDB r6!, {r11}") <built-in>:0)
(insn (set (reg:SI 11 fp)
        (const_int 560 [0x230])) -1
(insn (set (reg:SI 11 fp)
        (plus:SI (reg:SI 11 fp)
(insn (set (reg:CC 100 cc)
        (compare:CC (reg:SI 11 fp)
(insn (set (pc)
        (if_then_else (ne (reg:CC 100 cc)
(insn/f 66 94 67 (parallel [
            (set (mem/c:BLK (pre_modify:SI (reg/f:SI 13 sp)
(insn 7 67 85 (set (reg:DI 0 r0 [orig:110 _1 ] [110])
        (mem:DI (reg:SI 0 r0 [ n ]) [23 *n_4(D)+0 S8 A64]))  180 {*arm_movdi}
(insn 8 85 95 (parallel [
            (set (reg:CC_Z 100 cc)
(insn (cond_exec (eq (reg:CC 100 cc)
            (const_int 0 [0]))
(insn (cond_exec (ne (reg:CC 100 cc)
            (const_int 0 [0]))
(insn (set (reg:SI 11 fp)
        (plus:SI (reg:SI 11 fp)
(insn (set (reg:CC 100 cc)
        (compare:CC (reg:SI 11 fp)
(insn (set (pc)
        (if_then_else (ne (reg:CC 100 cc)
----------------------------------------
BB: 1
(insn (set (reg:SI 11 fp)
